In [54]:
import os
import torch
from torch.utils.data import Dataset

class MultiResCityscapesDataset(Dataset):
    def __init__(self, root='../data/msg', 
                 output_activation='tanh',
                 get_labels=False):
        """
        Args:
            root (str): The 'output_root' used in the preprocessing function.
            output_activation (str): 'tanh' or 'sigmoid' (used to build the path).
        """
        self.base_path = os.path.join(root, output_activation, "multi_res")
        self.images_dir = os.path.join(self.base_path, "images")
        self.labels_dir = os.path.join(self.base_path, "labels")
        self.get_labels = get_labels

        # Find all .pt files while maintaining order
        self.image_files = []
        for city in sorted(os.listdir(self.images_dir)):
            city_path = os.path.join(self.images_dir, city)
            if not os.path.isdir(city_path):
                continue
            for file in sorted(os.listdir(city_path)):
                if file.endswith(".pt"):
                    # Store relative path (e.g., 'berlin/berlin_000000_000019_leftImg8bit.pt')
                    self.image_files.append(os.path.join(city, file))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        rel_path = self.image_files[idx]
        
        # Load the list of tensors [4x4, 8x8, ..., 256x256]
        img_list = torch.load(os.path.join(self.images_dir, rel_path), weights_only=False)
        
        # Construct label path by replacing the suffix to match your preprocessing logic
        # Your script used: _leftImg8bit.png -> _gtCoarse_labelIds.png then .pt
        lbl_list = None
        if self.get_labels:
            label_rel_path = rel_path.replace("_leftImg8bit.pt", "_gtCoarse_labelIds.pt")
            label_path = os.path.join(self.labels_dir, label_rel_path)
            
            if os.path.exists(label_path):
                lbl_list = torch.load(label_path, weights_only=False)

        return img_list, lbl_list

# --- Usage Example ---
if __name__ == "__main__":
    dataset = MultiResCityscapesDataset(root='CityScapes/data/msg',
                                        output_activation='tanh',
                                        get_labels=True)
    
    # img_list will contain 7 tensors (4x4 up to 256x256)
    img_list, lbl_list = dataset[400]
    
    print(f"Number of scales: {len(img_list)}")
    print(f"Smallest scale shape: {img_list[0].shape}")  # torch.Size([3, 4, 4])
    print(f"Largest scale shape: {img_list[-1].shape}") # torch.Size([3, 256, 256])
    print(lbl_list)

Number of scales: 7
Smallest scale shape: torch.Size([3, 4, 4])
Largest scale shape: torch.Size([3, 256, 256])
None


In [1]:
import torch 
import torch.nn as nn



from CityScapes.datasets import MSGBundledDataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.optim import Adam
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


from refined_progan.blocks import L2NormConv2d
from refined_progan.blocks import ConvBlock
from refined_progan.blocks import UpsampleBlock

class Generator(nn.Module):
    def __init__(self, 
                 z_dim,
                channels,
                img_channels=3):
        super().__init__()
        self.z_dim = z_dim  
        self.channels = channels.copy()
        self.img_channels = img_channels

        # 1. Define the initial convolution layer
        self.initial_conv = nn.Sequential(
                            ConvBlock(in_channels=z_dim, 
                                out_channels=channels[0], 
                                kernel_size=4, 
                                stride=1, 
                                padding=0, 
                                bias=True,
                                transpose=True
                            ),
                            ConvBlock(in_channels=channels[0], 
                                out_channels=channels[0], 
                                kernel_size=3, 
                                stride=1, 
                                padding=1, 
                                bias=True,
                            )
        )

        self.blocks = nn.ModuleList([self.initial_conv])
        for i in range(1,len(self.channels)):
            self.blocks.append(
               UpsampleBlock(channels[i-1],channels[i])
            )


        self.rgb_layers = nn.ModuleList()
        for i in range(len(self.channels)):
            self.rgb_layers.append(
               L2NormConv2d(channels[i],self.img_channels,1,1,0)
            )
        self.activation = nn.Tanh()
     
    
    def forward(self, x):
        out  = []
        for i in range(len(self.channels)):
            x = self.blocks[i](x)
            y = self.rgb_layers[i](x)
            y = self.activation(y)
            out.append(y)
            
        return out

x  = torch.randn(4,100,1,1)
gen = Generator(100,[512,256,128,64,64,64,64])
y= gen(x)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


from refined_progan.blocks import L2NormConv2d
from refined_progan.blocks import ConvBlock
from refined_progan.blocks import DownsampleBlock

class Discriminator(nn.Module):
    def __init__(self, 
                channels,
                img_channels=3):
        super().__init__()
        self.channels = channels.copy()
        self.channels.reverse()
        self.img_channels = img_channels

        self.initial_block = DownsampleBlock(self.img_channels,self.channels[0])
        self.blocks = nn.ModuleList()
        for i in range(1,len(self.channels)-1):
            self.blocks.append(
               DownsampleBlock(self.channels[i-1]+self.img_channels,self.channels[i])
            )

        self.final_block =DownsampleBlock(self.channels[-2]+self.img_channels+1,
                                          self.channels[-1],
                                          kernel_size=4)
    
    def minibatch_std(self, x):
        batch_statistics =torch.std(x,dim=0).mean().repeat(x.shape[0],1,x.shape[2],x.shape[3])
        return torch.cat([x,batch_statistics],dim=1)
    def forward(self, x):
        y= self.initial_block(x[-1])
        for i in range(len(self.channels)-2):
            y = torch.cat([y,x[-i-2]],dim=1)
            y = self.blocks[i](y)

        y = torch.cat([y,x[0]],dim=1)
        y = self.minibatch_std(y)
        y = self.final_block(y)
        return y



In [4]:
z_dim =512
channels = [512,256,128,64,32,32,32]
batch_size =8

In [5]:
dataset = MSGBundledDataset(root='./CityScapes/data/progan')
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)


In [9]:

gen = Generator(z_dim,channels).to(device)
disc = Discriminator(channels).to(device)

gen_optimizer = Adam(gen.parameters(), lr=1e-4, betas=(0.5, 0.999))
disc_optimizer = Adam(disc.parameters(), lr=1e-4, betas=(0.5, 0.999))
noise =  torch.randn(1,z_dim,1,1).to(device)
fake = gen(noise)
disc_fake = disc(fake)

real = next(iter(loader))
disc_real = disc(real)

/tmp/ipykernel_1062845/2218485073.py:31: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  batch_statistics =torch.std(x,dim=0).mean().repeat(x.shape[0],1,x.shape[2],x.shape[3])


KeyError: -1